# Introduction

This notebook demonstrates a Bayesian energy fitting code using simulated single pulse data.

Assume single pulse energies $E$ are modeled as being drawn from a probability distribution $\mathcal{P}(E|\{\alpha\})$, described by parameters $\{\alpha\}$. If all pulse energies are independent of one another, the likelihood of a collection of energies $\{E\}=\{E_1,E_2,...,E_N\}$ is
\begin{equation}
\mathcal{L}(\{E\}|\{\alpha\})=\prod_{i=1}^N\mathcal{P}(E_i|\{\alpha\})
\end{equation}
For example, assume the pulses are described by a log-normal distribution with parameters $\{\alpha\}=\{\mu,\sigma\}$, with probability distribution function
\begin{equation}
\mathcal{P}_{\mathrm{LN}}(E|\mu,\sigma)=\frac{1}{\sqrt{2\pi}\sigma E}\exp\left[-\frac{(\ln E-\mu)^2}{2\sigma^2}\right]
\end{equation}
Then
\begin{equation}
\mathcal{L}_{\mathrm{LN}}(\{E\}|\mu,\sigma)=\prod_{i=1}^n\mathcal{P}_{\mathrm{LN}}(E_i|\mu,\sigma)=\left(\sqrt{2\pi}\sigma\right)^{-N}\prod_{i=1}^N\frac{1}{E_i}\exp\left[-\frac{(\ln E_i-\mu)^2}{2\sigma^2}\right]
\end{equation}
If we adopt suitable priors for $\mu$ and $\sigma$, we can derive posterior distributions for both parameters using Bayes' theorem:
\begin{equation}
\mathcal{P}(\{\alpha\}|\{E\})\propto\mathcal{L}(\{E\}|\{\alpha\})\mathcal{P}(\{\alpha\})
\end{equation}
We pick uniform priors for $\mu$ and log-uniform priors for $\sigma$, meaning that our priors are
\begin{equation}
\mathcal{P}(\mu,\sigma)\propto\frac{1}{\sigma}
\end{equation}
Our posteriors are then
\begin{equation}
\mathcal{P}_{\mathrm{LN}}(\mu,\sigma|\{E\})\propto\left(\sqrt{2\pi}\right)^{-N}\sigma^{-N-1}\prod_{i=1}^N\frac{1}{E_i}\exp\left[-\frac{(\ln E_i-\mu)^2}{2\sigma^2}\right]
\end{equation}

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.special as special

import corner
import emcee

import harmonic as hm

np.random.seed(0)

In [ ]:
def log_normal_pdf(E, mu, log_sig):
    """Log-normal probability distribution function"""
    sig = np.exp(log_sig)
    return (1/(np.sqrt(2*np.pi)*sig*E)) * np.exp(-(np.log(E) - mu)**2/(2*sig**2))

def log_normal_cdf(E, mu, log_sig):
    """Log-normal cumulative distribution function"""
    sig = np.exp(log_sig)
    return (1/2)*(1 + special.erf((np.log(E) - mu)/(np.sqrt(2)*sig)))

def gen_energy(mu, log_sig):
    """Draws energies from a log-normal distribution by picking a random
       number between 0 and 1, inverting the CDF, and solving"""
    u = np.random.uniform(0, 1)
    return np.exp(np.sqrt(2)*special.erfinv(2*u - 1) + mu)

# pulse parameters, chosen so that the mean of the pulse energies is 1
# and each of mu and sigma take on realistic values
mu = -0.5
sig = 1

N_pulses = 200
Es = np.array([gen_energy(mu, np.log(sig)) for i in range(N_pulses)])

We perform the fitting using a Markov Chain Monte Carlo algorithm implemented in the `emcee` package. We fit for $\ln\sigma$ instead of $\sigma$ to avoid numerical problems.

We also compute the evidence, which can be used for model comparison through Bayes factors. We do so using the learned harmonic mean estimator, implemented in the `harmonic` package. 

In [ ]:
# Define the log-likelihood
def log_likelihood(theta, Es):
    mu, log_sig = theta
    sig = np.exp(log_sig)
    return np.sum(-(1/2)*np.log(2*np.pi) - np.log(Es) - np.log(sig) - (np.log(Es) - mu)**2/(2*sig**2))

# Define the log-prior
def log_prior(theta):
    mu, log_sig = theta
    if -3 < mu < 3 and -2 < log_sig < 2:  # Uniform priors
        return 1/24
    return -np.inf  # Log(0) = -inf for disallowed values

# Define the log-posterior
def log_posterior(theta, Es):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, Es)

In [ ]:
ndim, nwalkers = 2, 100
pos = [[0, 0] + 1e-4*np.random.randn(ndim) for i in range(nwalkers)]

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_posterior, args=([Es]))
sampler.run_mcmc(pos, 10000)

samples = sampler.get_chain()[:, 50:, :].reshape((-1, ndim))
ln_flat_samples = sampler.get_chain(discard=100, thin=15, flat=True)

tot = 0
for i in range(len(ln_flat_samples)):
    mu_s, log_sig_s = ln_flat_samples[i]
    tot += np.exp(-log_likelihood([mu_s, log_sig_s], Es))
log_m = -np.log(tot) + np.log(len(ln_flat_samples))

nburn = 100
h_samples = np.ascontiguousarray(sampler.chain[:,nburn:,:])
h_lnprob = np.ascontiguousarray(sampler.lnprobability[:,nburn:])

chains = hm.Chains(ndim)
chains.add_chains_3d(h_samples, h_lnprob)
chains_train, chains_infer = hm.utils.split_data(chains, training_proportion=0.5)

n_scaled_layers = 2
n_unscaled_layers = 4
temperature = 0.8
    
model = hm.model.RealNVPModel(ndim, standardize=True, temperature=temperature)
epochs_num = 20
# Train model
model.fit(chains_train.samples, epochs=epochs_num, verbose= True)

# Instantiate harmonic's evidence class
ev = hm.Evidence(chains_infer.nchains, model)
    
# Pass the evidence class the inference chains and compute the evidence!
ev.add_chains(chains_infer)
ln_inv_evidence = ev.ln_evidence_inv
err_ln_inv_evidence = ev.compute_ln_inv_evidence_errors()

In [ ]:
fig = corner.corner(ln_flat_samples,
                    labels=[r'$\mu$', r'$\ln\sigma$'],
                    truths=[mu, np.log(sig)],
                    show_titles=True)
plt.show()

We can also plot a histogram of the empirical pulse energy probability distribution, as well as an curve predicting the number of pulses in each bin based on the medians of the posteriors for $\mu$ and $\ln\sigma$. All errors assume Poisson uncertainty, i.e. $\sigma_N=\sqrt{N}$, which holds for large $N$ but breaks down for bins with few pulses.

In [ ]:
nbins = 20
hist = plt.hist(np.log(Es), bins=nbins, color='k', histtype='step', label='Data with Poisson uncertainty')

nums = hist[0]
edges = hist[1]
bin_width = np.diff(edges)[0]
centers = edges[:-1] + bin_width/2

plt.errorbar(centers, nums, yerr=np.sqrt(nums), color='k', fmt='o', capsize=5)

mu_median = np.percentile(ln_flat_samples[:, 0], [16, 50, 84])[1]
log_sig_median = np.percentile(ln_flat_samples[:, 1], [16, 50, 84])[1]

log_Es_linspace = np.linspace(0.9*np.min(np.log(Es)), 1.1*np.max(np.log(Es)), 1000)
plt.plot(log_Es_linspace, log_normal_pdf(np.exp(log_Es_linspace), mu_median, log_sig_median)*bin_width*N_pulses*np.exp(log_Es_linspace), label='Fit')
plt.xlabel(r'$\ln(E)$')
plt.ylabel('Number of pulses')
plt.legend()
plt.show()